In [19]:
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import scipy
sc.settings.verbosity = 3
sc.logging.print_header()
sc.set_figure_params(dpi=100, dpi_save=600)
    
import matplotlib.pyplot as pl
from matplotlib import rcParams

import os

scanpy==1.10.4 anndata==0.11.3 umap==0.5.7 numpy==1.26.4 scipy==1.15.1 pandas==2.2.3 scikit-learn==1.6.1 statsmodels==0.14.4 igraph==0.11.8 pynndescent==0.5.13


In [2]:
# Samples with <100 cells were eliminated (C119_N_0_2_0_c1_v2, 60 cells)

# Define sample groups with their respective base directories and path formats
sample_groups = [
    {
        "base_dir": "/mnt/e/GSE236698/Matrix/scRNA-seq/",
        "samples": ['P1_N', 'P2_N', 'P3_N', 'P4_N', 'P5_N', 'P6_N',
                    'P1_T', 'P2_T', 'P3_T', 'P4_T', 'P5_T', 'P6_T'],
        "path_format": "{base_dir}{sample_id}/outs/{sample_id}_SoupX_Scrublet.h5ad"
    },
    {
        "base_dir": "/mnt/h/EGAD00001005191_EGAD00001005198/Matrix_v2024/scRNA-seq/EGAD00001005198/",
        "samples": ['SMC01N', 'SMC02N', 'SMC03N', 'SMC04N', 'SMC05N', 'SMC06N', 'SMC07N', 'SMC08N', 'SMC09N', 'SMC10N',
                    'SMC01T', 'SMC02T', 'SMC03T', 'SMC04T', 'SMC05T', 'SMC06T', 'SMC07T', 'SMC08T', 'SMC09T', 'SMC10T', 'SMC11T', 'SMC14T', 'SMC15T', 'SMC16T', 'SMC18T', 'SMC19T', 'SMC21T', 'SMC22T', 'SMC23T', 'SMC24T', 'SMC25T',
                    'SMC17T', 'SMC20T'],
        "path_format": "{base_dir}{sample_id}/outs/{sample_id}_SoupX_Scrublet.h5ad"
    }
]

# Function to read and process each sample
def read_sample(sample_id, base_dir, path_format):
    """Read sample data using the specified path format"""
    try:
        # Check if path_format requires a prefix
        if '{sample_prefix}' in path_format:
            sample_prefix = sample_id[:4]  # Extract first 4 characters
            path = path_format.format(base_dir=base_dir, sample_id=sample_id, sample_prefix=sample_prefix)
        else:
            path = path_format.format(base_dir=base_dir, sample_id=sample_id)
            
        adata = sc.read_h5ad(path)
        adata.obs["sample"] = sample_id
        adata.obs_names = [f"{sample_id}_{cell}" for cell in adata.obs_names]
        print(f"Successfully read {sample_id} | Cells: {adata.n_obs}, Genes: {adata.n_vars}")
        return adata
    except Exception as e:
        print(f"Failed to read {sample_id}: {e}")
        return None

# Read all samples into a list
adatas = []

# Read samples from all groups
for group in sample_groups:
    base_dir = group["base_dir"]
    path_format = group["path_format"]
    
    for sample in group["samples"]:
        adata = read_sample(sample, base_dir, path_format)
        if adata is not None:
            adatas.append(adata)

# Check if all datasets have identical gene sets
def check_genes_match(adatas_list):
    """Verify if all AnnData objects have identical gene sets"""
    first_genes = adatas_list[0].var_names.tolist()
    for adata in adatas_list[1:]:
        if adata.var_names.tolist() != first_genes:
            return False
    return True

# Merge datasets using compatible parameters for older Scanpy versions
if check_genes_match(adatas):
    print("All samples have identical gene sets. Performing direct concatenation.")
    combined_adata = sc.concat(
        adatas,
        axis='obs',           # Concatenate along observations (cells)
        join='inner',         # Use intersection of features (genes)
        merge='unique',       # Keep only unique metadata columns
        label='sample',       # Column in .obs to store batch information
        keys=[adata.obs["sample"][0] for adata in adatas],  # Batch categories
        index_unique='_'      # Make index unique by appending sample ID
    )
else:
    print("Gene sets differ between samples. Finding common genes...")
    common_genes = set(adatas[0].var_names)
    for adata in adatas[1:]:
        common_genes.intersection_update(adata.var_names)
    common_genes = list(common_genes)
    print(f"Found {len(common_genes)} common genes across all samples.")
    
    # Subset each AnnData to common genes before merging
    adatas_common = [adata[:, common_genes].copy() for adata in adatas]
    combined_adata = sc.concat(
        adatas_common,
        axis='obs',
        join='inner',
        merge='unique',
        label='sample',
        keys=[adata.obs["sample"][0] for adata in adatas_common],
        index_unique='_'
    )

Successfully read P1_N | Cells: 9464, Genes: 38606
Successfully read P2_N | Cells: 13850, Genes: 38606
Successfully read P3_N | Cells: 8845, Genes: 38606
Successfully read P4_N | Cells: 8429, Genes: 38606
Successfully read P5_N | Cells: 7614, Genes: 38606
Successfully read P6_N | Cells: 4627, Genes: 38606
Successfully read P1_T | Cells: 9334, Genes: 38606
Successfully read P2_T | Cells: 7404, Genes: 38606
Successfully read P3_T | Cells: 9406, Genes: 38606
Successfully read P4_T | Cells: 7822, Genes: 38606
Successfully read P5_T | Cells: 6568, Genes: 38606
Successfully read P6_T | Cells: 5585, Genes: 38606
Successfully read SMC01N | Cells: 4323, Genes: 38606
Successfully read SMC02N | Cells: 2315, Genes: 38606
Successfully read SMC03N | Cells: 1951, Genes: 38606
Successfully read SMC04N | Cells: 3150, Genes: 38606
Successfully read SMC05N | Cells: 2228, Genes: 38606
Successfully read SMC06N | Cells: 2968, Genes: 38606
Successfully read SMC07N | Cells: 3510, Genes: 38606
Successfully rea

/tmp/ipykernel_2171580/176006069.py:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  keys=[adata.obs["sample"][0] for adata in adatas],  # Batch categories


In [3]:
combined_adata

AnnData object with n_obs × n_vars = 202684 × 38606
    obs: 'doublet_score', 'predicted_doublet', 'sample'
    var: 'gene_ids'

In [4]:
combined_adata.obs['sample']

P1_N_AAACCCAAGAGTGTTA-1_P1_N          P1_N
P1_N_AAACCCAAGCCATTTG-1_P1_N          P1_N
P1_N_AAACCCAAGGGTATAT-1_P1_N          P1_N
P1_N_AAACCCAAGTAAATGC-1_P1_N          P1_N
P1_N_AAACCCACACCAAATC-1_P1_N          P1_N
                                     ...  
SMC20T_TTTGTCACAGATCTGT-1_SMC20T    SMC20T
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T    SMC20T
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T    SMC20T
SMC20T_TTTGTCATCACGCGGT-1_SMC20T    SMC20T
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T    SMC20T
Name: sample, Length: 202684, dtype: category
Categories (45, object): ['P1_N', 'P2_N', 'P3_N', 'P4_N', ..., 'SMC24T', 'SMC25T', 'SMC17T', 'SMC20T']

In [5]:
combined_adata.obs['pathology'] = (combined_adata.obs['sample'].map
                          (lambda x: {'P1_N':'Normal', 'P2_N':'Normal', 'P3_N':'Normal', 'P4_N':'Normal', 'P5_N':'Normal', 'P6_N':'Normal',
                                      'P1_T':'MCAD', 'P2_T':'MCAD', 'P3_T':'MCAD', 'P4_T':'MCAD', 'P5_T':'MCAD', 'P6_T':'MCAD',
                                      'SMC01N':'Normal', 'SMC02N':'Normal', 'SMC03N':'Normal', 'SMC04N':'Normal', 'SMC05N':'Normal', 'SMC06N':'Normal', 'SMC07N':'Normal', 'SMC08N':'Normal', 'SMC09N':'Normal', 'SMC10N':'Normal',
                                      'SMC01T':'NMCAD', 'SMC02T':'NMCAD', 'SMC03T':'NMCAD', 'SMC04T':'NMCAD', 'SMC05T':'NMCAD', 'SMC06T':'NMCAD', 'SMC07T':'NMCAD', 'SMC08T':'NMCAD', 'SMC09T':'NMCAD', 'SMC10T':'NMCAD', 'SMC11T':'NMCAD', 'SMC14T':'NMCAD', 'SMC15T':'NMCAD', 'SMC16T':'NMCAD', 'SMC18T':'NMCAD', 'SMC19T':'NMCAD', 'SMC21T':'NMCAD', 'SMC22T':'NMCAD', 'SMC23T':'NMCAD', 'SMC24T':'NMCAD', 'SMC25T':'NMCAD',
                                      'SMC17T':'MCAD', 'SMC20T':'MCAD'}.get(x, x)).astype("category"))

In [6]:
combined_adata.obs['pathology']

P1_N_AAACCCAAGAGTGTTA-1_P1_N        Normal
P1_N_AAACCCAAGCCATTTG-1_P1_N        Normal
P1_N_AAACCCAAGGGTATAT-1_P1_N        Normal
P1_N_AAACCCAAGTAAATGC-1_P1_N        Normal
P1_N_AAACCCACACCAAATC-1_P1_N        Normal
                                     ...  
SMC20T_TTTGTCACAGATCTGT-1_SMC20T      MCAD
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T      MCAD
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T      MCAD
SMC20T_TTTGTCATCACGCGGT-1_SMC20T      MCAD
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T      MCAD
Name: pathology, Length: 202684, dtype: category
Categories (3, object): ['MCAD', 'NMCAD', 'Normal']

In [7]:
combined_adata.obs['pathology'] = pd.Categorical(combined_adata.obs['pathology'], categories=['Normal', 'NMCAD', 'MCAD'])

In [8]:
combined_adata.obs['pathology']

P1_N_AAACCCAAGAGTGTTA-1_P1_N        Normal
P1_N_AAACCCAAGCCATTTG-1_P1_N        Normal
P1_N_AAACCCAAGGGTATAT-1_P1_N        Normal
P1_N_AAACCCAAGTAAATGC-1_P1_N        Normal
P1_N_AAACCCACACCAAATC-1_P1_N        Normal
                                     ...  
SMC20T_TTTGTCACAGATCTGT-1_SMC20T      MCAD
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T      MCAD
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T      MCAD
SMC20T_TTTGTCATCACGCGGT-1_SMC20T      MCAD
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T      MCAD
Name: pathology, Length: 202684, dtype: category
Categories (3, object): ['Normal', 'NMCAD', 'MCAD']

In [9]:
combined_adata.obs['patient'] = (combined_adata.obs['sample'].map
                          (lambda x: {'P1_N':'P1N', 'P2_N':'P2N', 'P3_N':'P3N', 'P4_N':'P4N', 'P5_N':'P5N', 'P6_N':'P6N',
                                      'P1_T':'P1TM', 'P2_T':'P2TM', 'P3_T':'P3TM', 'P4_T':'P4TM', 'P5_T':'P5TM', 'P6_T':'P6TM',
                                      'SMC01N':'SMC01N', 'SMC02N':'SMC02N', 'SMC03N':'SMC03N', 'SMC04N':'SMC04N', 'SMC05N':'SMC05N', 'SMC06N':'SMC06N', 'SMC07N':'SMC07N', 'SMC08N':'SMC08N', 'SMC09N':'SMC09N', 'SMC10N':'SMC10N',
                                      'SMC01T':'SMC01TN', 'SMC02T':'SMC02TN', 'SMC03T':'SMC03TN', 'SMC04T':'SMC04TN', 'SMC05T':'SMC05TN', 'SMC06T':'SMC06TN', 'SMC07T':'SMC07TN', 'SMC08T':'SMC08TN', 'SMC09T':'SMC09TN', 'SMC10T':'SMC10TN', 'SMC11T':'SMC11TN', 'SMC14T':'SMC14TN', 'SMC15T':'SMC15TN', 'SMC16T':'SMC16TN', 'SMC18T':'SMC18TN', 'SMC19T':'SMC19TN', 'SMC21T':'SMC21TN', 'SMC22T':'SMC22TN', 'SMC23T':'SMC23TN', 'SMC24T':'SMC24TN', 'SMC25T':'SMC25TN',
                                      'SMC17T':'SMC17TM', 'SMC20T':'SMC20TM'}.get(x, x)).astype("category"))

In [10]:
combined_adata.obs['patient']

P1_N_AAACCCAAGAGTGTTA-1_P1_N            P1N
P1_N_AAACCCAAGCCATTTG-1_P1_N            P1N
P1_N_AAACCCAAGGGTATAT-1_P1_N            P1N
P1_N_AAACCCAAGTAAATGC-1_P1_N            P1N
P1_N_AAACCCACACCAAATC-1_P1_N            P1N
                                     ...   
SMC20T_TTTGTCACAGATCTGT-1_SMC20T    SMC20TM
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T    SMC20TM
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T    SMC20TM
SMC20T_TTTGTCATCACGCGGT-1_SMC20T    SMC20TM
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T    SMC20TM
Name: patient, Length: 202684, dtype: category
Categories (45, object): ['P1N', 'P2N', 'P3N', 'P4N', ..., 'SMC24TN', 'SMC25TN', 'SMC17TM', 'SMC20TM']

In [11]:
combined_adata.obs['dataset'] = (combined_adata.obs['sample'].map
                          (lambda x: {'P1_N':'GSE236698', 'P2_N':'GSE236698', 'P3_N':'GSE236698', 'P4_N':'GSE236698', 'P5_N':'GSE236698', 'P6_N':'GSE236698',
                                      'P1_T':'GSE236698', 'P2_T':'GSE236698', 'P3_T':'GSE236698', 'P4_T':'GSE236698', 'P5_T':'GSE236698', 'P6_T':'GSE236698',
                                      'SMC01N':'SMC', 'SMC02N':'SMC', 'SMC03N':'SMC', 'SMC04N':'SMC', 'SMC05N':'SMC', 'SMC06N':'SMC', 'SMC07N':'SMC', 'SMC08N':'SMC', 'SMC09N':'SMC', 'SMC10N':'SMC',
                                      'SMC01T':'SMC', 'SMC02T':'SMC', 'SMC03T':'SMC', 'SMC04T':'SMC', 'SMC05T':'SMC', 'SMC06T':'SMC', 'SMC07T':'SMC', 'SMC08T':'SMC', 'SMC09T':'SMC', 'SMC10T':'SMC', 'SMC11T':'SMC', 'SMC14T':'SMC', 'SMC15T':'SMC', 'SMC16T':'SMC', 'SMC18T':'SMC', 'SMC19T':'SMC', 'SMC21T':'SMC', 'SMC22T':'SMC', 'SMC23T':'SMC', 'SMC24T':'SMC', 'SMC25T':'SMC',
                                      'SMC17T':'SMC', 'SMC20T':'SMC'}.get(x, x)).astype("category"))

In [12]:
combined_adata.obs['dataset']

P1_N_AAACCCAAGAGTGTTA-1_P1_N        GSE236698
P1_N_AAACCCAAGCCATTTG-1_P1_N        GSE236698
P1_N_AAACCCAAGGGTATAT-1_P1_N        GSE236698
P1_N_AAACCCAAGTAAATGC-1_P1_N        GSE236698
P1_N_AAACCCACACCAAATC-1_P1_N        GSE236698
                                      ...    
SMC20T_TTTGTCACAGATCTGT-1_SMC20T          SMC
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T          SMC
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T          SMC
SMC20T_TTTGTCATCACGCGGT-1_SMC20T          SMC
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T          SMC
Name: dataset, Length: 202684, dtype: category
Categories (2, object): ['GSE236698', 'SMC']

In [13]:
# Save merged data
output_path = f"/mnt/c/Users/hyj/Downloads/SoupX_Scrublet_Merge.h5ad"
combined_adata.write(output_path, compression="gzip")
print(f"Merged data saved to: {output_path}")
print(f"Final dataset contains {combined_adata.n_obs} cells and {combined_adata.n_vars} genes.")

Merged data saved to: /mnt/c/Users/hyj/Downloads/SoupX_Scrublet_Merge.h5ad
Final dataset contains 202684 cells and 38606 genes.


In [26]:
Normal=combined_adata[combined_adata.obs['pathology'].isin(['Normal'])]
Normal

View of AnnData object with n_obs × n_vars = 80977 × 38606
    obs: 'doublet_score', 'predicted_doublet', 'sample', 'pathology', 'patient', 'dataset'
    var: 'gene_ids'

In [27]:
Normal.obs['sample']

P1_N_AAACCCAAGAGTGTTA-1_P1_N          P1_N
P1_N_AAACCCAAGCCATTTG-1_P1_N          P1_N
P1_N_AAACCCAAGGGTATAT-1_P1_N          P1_N
P1_N_AAACCCAAGTAAATGC-1_P1_N          P1_N
P1_N_AAACCCACACCAAATC-1_P1_N          P1_N
                                     ...  
SMC10N_TTTGTCACACTGTGTA-1_SMC10N    SMC10N
SMC10N_TTTGTCAGTAAATGAC-1_SMC10N    SMC10N
SMC10N_TTTGTCAGTACCTACA-1_SMC10N    SMC10N
SMC10N_TTTGTCAGTCACCTAA-1_SMC10N    SMC10N
SMC10N_TTTGTCAGTTCCAACA-1_SMC10N    SMC10N
Name: sample, Length: 80977, dtype: category
Categories (16, object): ['P1_N', 'P2_N', 'P3_N', 'P4_N', ..., 'SMC07N', 'SMC08N', 'SMC09N', 'SMC10N']

In [28]:
Normal.obs['patient']

P1_N_AAACCCAAGAGTGTTA-1_P1_N           P1N
P1_N_AAACCCAAGCCATTTG-1_P1_N           P1N
P1_N_AAACCCAAGGGTATAT-1_P1_N           P1N
P1_N_AAACCCAAGTAAATGC-1_P1_N           P1N
P1_N_AAACCCACACCAAATC-1_P1_N           P1N
                                     ...  
SMC10N_TTTGTCACACTGTGTA-1_SMC10N    SMC10N
SMC10N_TTTGTCAGTAAATGAC-1_SMC10N    SMC10N
SMC10N_TTTGTCAGTACCTACA-1_SMC10N    SMC10N
SMC10N_TTTGTCAGTCACCTAA-1_SMC10N    SMC10N
SMC10N_TTTGTCAGTTCCAACA-1_SMC10N    SMC10N
Name: patient, Length: 80977, dtype: category
Categories (16, object): ['P1N', 'P2N', 'P3N', 'P4N', ..., 'SMC07N', 'SMC08N', 'SMC09N', 'SMC10N']

In [29]:
Normal.obs['dataset']

P1_N_AAACCCAAGAGTGTTA-1_P1_N        GSE236698
P1_N_AAACCCAAGCCATTTG-1_P1_N        GSE236698
P1_N_AAACCCAAGGGTATAT-1_P1_N        GSE236698
P1_N_AAACCCAAGTAAATGC-1_P1_N        GSE236698
P1_N_AAACCCACACCAAATC-1_P1_N        GSE236698
                                      ...    
SMC10N_TTTGTCACACTGTGTA-1_SMC10N          SMC
SMC10N_TTTGTCAGTAAATGAC-1_SMC10N          SMC
SMC10N_TTTGTCAGTACCTACA-1_SMC10N          SMC
SMC10N_TTTGTCAGTCACCTAA-1_SMC10N          SMC
SMC10N_TTTGTCAGTTCCAACA-1_SMC10N          SMC
Name: dataset, Length: 80977, dtype: category
Categories (2, object): ['GSE236698', 'SMC']

In [30]:
Normal.write('/mnt/c/Users/hyj/Downloads/Normal.h5ad', compression="gzip")

In [31]:
NMCAD=combined_adata[combined_adata.obs['pathology'].isin(['NMCAD'])]
NMCAD

View of AnnData object with n_obs × n_vars = 70512 × 38606
    obs: 'doublet_score', 'predicted_doublet', 'sample', 'pathology', 'patient', 'dataset'
    var: 'gene_ids'

In [32]:
NMCAD.obs['sample']

SMC01T_AAACCTGAGAAGGTTT-1_SMC01T    SMC01T
SMC01T_AAACCTGAGGTAGCTG-1_SMC01T    SMC01T
SMC01T_AAACCTGAGTCGTACT-1_SMC01T    SMC01T
SMC01T_AAACCTGCATACGCCG-1_SMC01T    SMC01T
SMC01T_AAACCTGGTCATATCG-1_SMC01T    SMC01T
                                     ...  
SMC25T_TTTGTCAAGCGCTCCA-1_SMC25T    SMC25T
SMC25T_TTTGTCAAGCGTTCCG-1_SMC25T    SMC25T
SMC25T_TTTGTCAAGTTCGATC-1_SMC25T    SMC25T
SMC25T_TTTGTCACAAATACAG-1_SMC25T    SMC25T
SMC25T_TTTGTCATCCCAAGTA-1_SMC25T    SMC25T
Name: sample, Length: 70512, dtype: category
Categories (21, object): ['SMC01T', 'SMC02T', 'SMC03T', 'SMC04T', ..., 'SMC22T', 'SMC23T', 'SMC24T', 'SMC25T']

In [33]:
NMCAD.obs['patient']

SMC01T_AAACCTGAGAAGGTTT-1_SMC01T    SMC01TN
SMC01T_AAACCTGAGGTAGCTG-1_SMC01T    SMC01TN
SMC01T_AAACCTGAGTCGTACT-1_SMC01T    SMC01TN
SMC01T_AAACCTGCATACGCCG-1_SMC01T    SMC01TN
SMC01T_AAACCTGGTCATATCG-1_SMC01T    SMC01TN
                                     ...   
SMC25T_TTTGTCAAGCGCTCCA-1_SMC25T    SMC25TN
SMC25T_TTTGTCAAGCGTTCCG-1_SMC25T    SMC25TN
SMC25T_TTTGTCAAGTTCGATC-1_SMC25T    SMC25TN
SMC25T_TTTGTCACAAATACAG-1_SMC25T    SMC25TN
SMC25T_TTTGTCATCCCAAGTA-1_SMC25T    SMC25TN
Name: patient, Length: 70512, dtype: category
Categories (21, object): ['SMC01TN', 'SMC02TN', 'SMC03TN', 'SMC04TN', ..., 'SMC22TN', 'SMC23TN', 'SMC24TN', 'SMC25TN']

In [34]:
NMCAD.obs['dataset']

SMC01T_AAACCTGAGAAGGTTT-1_SMC01T    SMC
SMC01T_AAACCTGAGGTAGCTG-1_SMC01T    SMC
SMC01T_AAACCTGAGTCGTACT-1_SMC01T    SMC
SMC01T_AAACCTGCATACGCCG-1_SMC01T    SMC
SMC01T_AAACCTGGTCATATCG-1_SMC01T    SMC
                                   ... 
SMC25T_TTTGTCAAGCGCTCCA-1_SMC25T    SMC
SMC25T_TTTGTCAAGCGTTCCG-1_SMC25T    SMC
SMC25T_TTTGTCAAGTTCGATC-1_SMC25T    SMC
SMC25T_TTTGTCACAAATACAG-1_SMC25T    SMC
SMC25T_TTTGTCATCCCAAGTA-1_SMC25T    SMC
Name: dataset, Length: 70512, dtype: category
Categories (1, object): ['SMC']

In [35]:
NMCAD.write('/mnt/c/Users/hyj/Downloads/NMCAD.h5ad', compression="gzip")

In [14]:
MCAD=combined_adata[combined_adata.obs['pathology'].isin(['MCAD'])]
MCAD

View of AnnData object with n_obs × n_vars = 51195 × 38606
    obs: 'doublet_score', 'predicted_doublet', 'sample', 'pathology', 'patient', 'dataset'
    var: 'gene_ids'

In [15]:
MCAD.obs['sample']

P1_T_AAACCCAAGCTCATAC-1_P1_T          P1_T
P1_T_AAACCCAAGGATATGT-1_P1_T          P1_T
P1_T_AAACCCAAGTATAACG-1_P1_T          P1_T
P1_T_AAACCCACAAATGGCG-1_P1_T          P1_T
P1_T_AAACCCACAACTAGAA-1_P1_T          P1_T
                                     ...  
SMC20T_TTTGTCACAGATCTGT-1_SMC20T    SMC20T
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T    SMC20T
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T    SMC20T
SMC20T_TTTGTCATCACGCGGT-1_SMC20T    SMC20T
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T    SMC20T
Name: sample, Length: 51195, dtype: category
Categories (8, object): ['P1_T', 'P2_T', 'P3_T', 'P4_T', 'P5_T', 'P6_T', 'SMC17T', 'SMC20T']

In [16]:
MCAD.obs['patient']

P1_T_AAACCCAAGCTCATAC-1_P1_T           P1TM
P1_T_AAACCCAAGGATATGT-1_P1_T           P1TM
P1_T_AAACCCAAGTATAACG-1_P1_T           P1TM
P1_T_AAACCCACAAATGGCG-1_P1_T           P1TM
P1_T_AAACCCACAACTAGAA-1_P1_T           P1TM
                                     ...   
SMC20T_TTTGTCACAGATCTGT-1_SMC20T    SMC20TM
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T    SMC20TM
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T    SMC20TM
SMC20T_TTTGTCATCACGCGGT-1_SMC20T    SMC20TM
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T    SMC20TM
Name: patient, Length: 51195, dtype: category
Categories (8, object): ['P1TM', 'P2TM', 'P3TM', 'P4TM', 'P5TM', 'P6TM', 'SMC17TM', 'SMC20TM']

In [17]:
MCAD.obs['dataset']

P1_T_AAACCCAAGCTCATAC-1_P1_T        GSE236698
P1_T_AAACCCAAGGATATGT-1_P1_T        GSE236698
P1_T_AAACCCAAGTATAACG-1_P1_T        GSE236698
P1_T_AAACCCACAAATGGCG-1_P1_T        GSE236698
P1_T_AAACCCACAACTAGAA-1_P1_T        GSE236698
                                      ...    
SMC20T_TTTGTCACAGATCTGT-1_SMC20T          SMC
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T          SMC
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T          SMC
SMC20T_TTTGTCATCACGCGGT-1_SMC20T          SMC
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T          SMC
Name: dataset, Length: 51195, dtype: category
Categories (2, object): ['GSE236698', 'SMC']

In [18]:
MCAD.write('/mnt/c/Users/hyj/Downloads/MCAD.h5ad', compression="gzip")

In [36]:
NMCAD_MCAD=combined_adata[combined_adata.obs['pathology'].isin(['NMCAD', 'MCAD'])]
NMCAD_MCAD

View of AnnData object with n_obs × n_vars = 121707 × 38606
    obs: 'doublet_score', 'predicted_doublet', 'sample', 'pathology', 'patient', 'dataset'
    var: 'gene_ids'

In [37]:
NMCAD_MCAD.obs['sample']

P1_T_AAACCCAAGCTCATAC-1_P1_T          P1_T
P1_T_AAACCCAAGGATATGT-1_P1_T          P1_T
P1_T_AAACCCAAGTATAACG-1_P1_T          P1_T
P1_T_AAACCCACAAATGGCG-1_P1_T          P1_T
P1_T_AAACCCACAACTAGAA-1_P1_T          P1_T
                                     ...  
SMC20T_TTTGTCACAGATCTGT-1_SMC20T    SMC20T
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T    SMC20T
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T    SMC20T
SMC20T_TTTGTCATCACGCGGT-1_SMC20T    SMC20T
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T    SMC20T
Name: sample, Length: 121707, dtype: category
Categories (29, object): ['P1_T', 'P2_T', 'P3_T', 'P4_T', ..., 'SMC24T', 'SMC25T', 'SMC17T', 'SMC20T']

In [38]:
NMCAD_MCAD.obs['patient']

P1_T_AAACCCAAGCTCATAC-1_P1_T           P1TM
P1_T_AAACCCAAGGATATGT-1_P1_T           P1TM
P1_T_AAACCCAAGTATAACG-1_P1_T           P1TM
P1_T_AAACCCACAAATGGCG-1_P1_T           P1TM
P1_T_AAACCCACAACTAGAA-1_P1_T           P1TM
                                     ...   
SMC20T_TTTGTCACAGATCTGT-1_SMC20T    SMC20TM
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T    SMC20TM
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T    SMC20TM
SMC20T_TTTGTCATCACGCGGT-1_SMC20T    SMC20TM
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T    SMC20TM
Name: patient, Length: 121707, dtype: category
Categories (29, object): ['P1TM', 'P2TM', 'P3TM', 'P4TM', ..., 'SMC24TN', 'SMC25TN', 'SMC17TM', 'SMC20TM']

In [39]:
NMCAD_MCAD.obs['dataset']

P1_T_AAACCCAAGCTCATAC-1_P1_T        GSE236698
P1_T_AAACCCAAGGATATGT-1_P1_T        GSE236698
P1_T_AAACCCAAGTATAACG-1_P1_T        GSE236698
P1_T_AAACCCACAAATGGCG-1_P1_T        GSE236698
P1_T_AAACCCACAACTAGAA-1_P1_T        GSE236698
                                      ...    
SMC20T_TTTGTCACAGATCTGT-1_SMC20T          SMC
SMC20T_TTTGTCAGTACCGTAT-1_SMC20T          SMC
SMC20T_TTTGTCAGTCAAACTC-1_SMC20T          SMC
SMC20T_TTTGTCATCACGCGGT-1_SMC20T          SMC
SMC20T_TTTGTCATCGCCAAAT-1_SMC20T          SMC
Name: dataset, Length: 121707, dtype: category
Categories (2, object): ['GSE236698', 'SMC']

In [18]:
NMCAD_MCAD.write('/mnt/c/Users/hyj/Downloads/NMCAD_MCAD.h5ad', compression="gzip")